# Optimización de Campañas Publicitarias con Multi-Armed Bandits
### Thompson Sampling vs. A/B Testing, ε-greedy y UCB1

**Objetivo:** comparar de forma rigurosa (mediante simulación Monte Carlo) el rendimiento
de distintos algoritmos de asignación de tráfico en un escenario de marketing digital,
donde varias creatividades de anuncio compiten por conversión.

**Pregunta de investigación:** ¿cuánto *regret* acumulado genera cada algoritmo frente
a Thompson Sampling, y cómo cambia esa diferencia según la dificultad del escenario
(diferencias entre tasas de conversión, número de brazos, horizonte temporal)?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from dataclasses import dataclass, field

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

### 1. Entorno de simulación: `BanditEnvironment`
Modelamos cada anuncio (brazo) como una variable de Bernoulli con una probabilidad de
conversión real `p_i`, desconocida para los algoritmos.

In [2]:
@dataclass
class BanditEnvironment:
    true_probs: list[float]

    def __post_init__(self):
        self.n_arms = len(self.true_probs)
        self.true_probs = np.array(self.true_probs)
        self.best_arm = np.argmax(self.true_probs)
        self.best_prob = self.true_probs[self.best_arm]

    def pull(self, arm_index: int) -> int:
        """Simula mostrar el anuncio `arm_index` a un usuario y devuelve 1 (conversión) o 0."""
        return int(np.random.rand() < self.true_probs[arm_index])

In [3]:
env_test = BanditEnvironment(true_probs=[0.05, 0.08, 0.03])

print(f"Número de brazos: {env_test.n_arms}")
print(f"Mejor brazo: {env_test.best_arm} (p = {env_test.best_prob})")
print(f"10 tiradas del brazo 1: {[env_test.pull(1) for _ in range(10)]}")

Número de brazos: 3
Mejor brazo: 1 (p = 0.08)
10 tiradas del brazo 1: [0, 0, 0, 0, 0, 0, 1, 0, 0, 0]


### 2. Escenarios predefinidos

Para poder comparar los algoritmos de forma rigurosa, definimos varios escenarios con
distinta dificultad: diferencias grandes vs. pequeñas entre las tasas de conversión de
los brazos, y pocos vs. muchos brazos. Los añadimos como `classmethod` de
`BanditEnvironment` para no tener que escribir las probabilidades a mano cada vez que
los usemos.

In [ ]:
@dataclass
class BanditEnvironment:
    true_probs: list[float]

    def __post_init__(self):
        self.n_arms = len(self.true_probs)
        self.true_probs = np.array(self.true_probs)
        self.best_arm = np.argmax(self.true_probs)
        self.best_prob = self.true_probs[self.best_arm]

    def pull(self, arm_index: int) -> int:
        """Simula mostrar el anuncio `arm_index` a un usuario y devuelve 1 (conversión) o 0."""
        return int(np.random.rand() < self.true_probs[arm_index])

    @classmethod
    def escenario_facil(cls):
        """Diferencias grandes entre brazos"""
        return cls(true_probs=[0.02, 0.05, 0.1])

    @classmethod
    def escenario_dificil(cls):
        """Diferencias muy pequeñas entre brazos"""
        return cls(true_probs=[0.048, 0.050, 0.052])

    @classmethod
    def escenario_pocos_brazos(cls):
        """k=3, referencia base."""
        return cls(true_probs=[0.03, 0.05, 0.08])

    @classmethod
    def escenario_muchos_brazos(cls):
        """k=15, con un único brazo claramente mejor entre muchos mediocres."""
        rng = np.random.default_rng(7)
        probs = rng.uniform(0.02, 0.05, size=14).tolist()
        probs.append(0.09)  # el brazo ganador, al final de la lista
        return cls(true_probs=probs)

In [7]:
escenarios = {
    "Fácil": BanditEnvironment.escenario_facil(),
    "Difícil": BanditEnvironment.escenario_dificil(),
    "Pocos brazos": BanditEnvironment.escenario_pocos_brazos(),
    "Muchos brazos": BanditEnvironment.escenario_muchos_brazos(),
}

for nombre, env in escenarios.items():
    print(f"{nombre}: k={env.n_arms}, probs={env.true_probs.round(3)}, mejor brazo={env.best_arm} (p={env.best_prob:.3f})")

Fácil: k=3, probs=[0.02 0.05 0.1 ], mejor brazo=2 (p=0.100)
Difícil: k=3, probs=[0.048 0.05  0.052], mejor brazo=2 (p=0.052)
Pocos brazos: k=3, probs=[0.03 0.05 0.08], mejor brazo=2 (p=0.080)
Muchos brazos: k=15, probs=[0.039 0.047 0.043 0.027 0.029 0.046 0.02  0.045 0.044 0.034 0.029 0.028
 0.028 0.033 0.09 ], mejor brazo=14 (p=0.090)


### 3. Algoritmos de asignación de tráfico

Implementamos cuatro estrategias distintas para decidir, en cada ronda, qué brazo
mostrar. Cada una devuelve una secuencia de decisiones y resultados que luego usaremos
para calcular el regret acumulado y comparar objetivamente su rendimiento.

#### 3.1 A/B Testing tradicional

El enfoque "de manual": se reparte el tráfico a partes iguales entre todos los brazos
durante toda la duración del experimento (fase de exploración pura), y solo al final
se decide cuál es el ganador para explotarlo. No hay adaptación durante el proceso.

In [9]:
def run_ab_testing(env: BanditEnvironment, T: int, split_ratio: float = None):
    """
    Simula A/B testing tradicional: reparte el tráfico a partes iguales entre
    todos los brazos durante toda la duración T.
    
    Devuelve:
        chosen_arms: array de longitud T con el índice del brazo elegido en cada ronda
        rewards: array de longitud T con el resultado (0/1) obtenido en cada ronda
    """
    chosen_arms = np.zeros(T, dtype=int)
    rewards = np.zeros(T, dtype=int)

    for t in range(T):
        arm = t % env.n_arms  # reparto round-robin: 0,1,2,0,1,2,...
        reward = env.pull(arm)

        chosen_arms[t] = arm
        rewards[t] = reward

    return chosen_arms, rewards

In [16]:
env = BanditEnvironment.escenario_facil()
chosen, rewards = run_ab_testing(env, T=1000)

print(f"Total de conversiones obtenidas: {rewards.sum()} de {len(rewards)} rondas")
print(f"Tasa de conversión observada: {rewards.mean():.4f}")
print(f"Reparto de tráfico por brazo: {np.bincount(chosen)}")

Total de conversiones obtenidas: 57 de 1000 rondas
Tasa de conversión observada: 0.0570
Reparto de tráfico por brazo: [334 333 333]


#### 3.2 ε-greedy

En cada ronda, con probabilidad ε se elige un brazo al azar (exploración), y con
probabilidad 1-ε se elige el brazo con mejor tasa de conversión observada hasta el
momento (explotación). A diferencia de A/B testing, aquí sí hay adaptación: el
algoritmo va aprendiendo de los resultados según avanza el experimento.

In [17]:
def run_epsilon_greedy(env: BanditEnvironment, T: int, epsilon: float = 0.1):
    """
    Simula ε-greedy: con probabilidad epsilon elige un brazo al azar,
    con probabilidad 1-epsilon elige el brazo con mejor media observada.

    Devuelve:
        chosen_arms: array de longitud T con el índice del brazo elegido en cada ronda
        rewards: array de longitud T con el resultado (0/1) obtenido en cada ronda
    """
    chosen_arms = np.zeros(T, dtype=int)
    rewards = np.zeros(T, dtype=int)

    counts = np.zeros(env.n_arms)      # nº de veces que se ha probado cada brazo
    sums = np.zeros(env.n_arms)        # suma de recompensas obtenidas por cada brazo

    for t in range(T):
        if np.random.rand() < epsilon:
            arm = np.random.randint(env.n_arms)   # exploración: brazo al azar
        else:
            means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
            arm = np.argmax(means)                 # explotación: mejor media hasta ahora

        reward = env.pull(arm)

        counts[arm] += 1
        sums[arm] += reward
        chosen_arms[t] = arm
        rewards[t] = reward

    return chosen_arms, rewards

In [24]:
env = BanditEnvironment.escenario_facil()
chosen, rewards = run_epsilon_greedy(env, T=1000, epsilon=0.1)

print(f"Total de conversiones obtenidas: {rewards.sum()} de {len(rewards)} rondas")
print(f"Tasa de conversión observada: {rewards.mean():.4f}")
print(f"Reparto de tráfico por brazo: {np.bincount(chosen, minlength=env.n_arms)}")
print(f"Mejor brazo real: {env.best_arm} — veces elegido: {np.bincount(chosen, minlength=env.n_arms)[env.best_arm]}")

Total de conversiones obtenidas: 86 de 1000 rondas
Tasa de conversión observada: 0.0860
Reparto de tráfico por brazo: [ 99  27 874]
Mejor brazo real: 2 — veces elegido: 874
